In [ ]:
# Imports
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load Dataset
BATCH_SIZE = 32
IMG_SIZE = (224,224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    "E:\datasets\PlantVillage",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "E:\datasets\PlantVillage",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
# Prefetch
AUTOTUNE= tf.data.AUTOTUNE
train_ds= train_ds.prefetch(AUTOTUNE)
val_ds= val_ds.prefetch(AUTOTUNE)

In [ ]:
# Loading Base-Model and Preprocessing Input
base_model = tf.keras.applications.MobileNetV2(
    input_shape = IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet')
base_model.trainable= False

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [ ]:
# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

In [ ]:
# Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',       
    patience=3,               
    restore_best_weights=True)

In [ ]:
# Build MobileNetV2 Frozen Model
NUM_CLASSES= 15
inputs = tf.keras.Input(shape=(224,224,3))
x= data_augmentation(inputs)
x= preprocess_input(x)
x= base_model(x, training= False)
x= tf.keras.layers.GlobalAveragePooling2D()(x)
x= tf.keras.layers.Dropout(0.2)(x)
outputs= tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model= tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'])

model.summary()

In [ ]:
# Train Frozen Model
checkpoint = ModelCheckpoint(
    filepath='frozen_model.keras', 
    monitor='val_loss',       
    save_best_only=True, 
    verbose=1)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, checkpoint])

In [ ]:
# Fine Tune Upper Layers
frozen_model = tf.keras.models.load_model("frozen_model.keras") #Load the already trained frozen model

mobilenet_model = frozen_model.get_layer("mobilenetv2_1.00_224") #Access the MobileNetV2 layer(model) in the frozen model
mobilenet_model.trainable= True                                   #Un-frozen for fine tuning

for layer in mobilenet_model.layers[:134]: #Freezing every layer below 134th
    layer.trainable = False

frozen_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   #lowering the learning rate
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"])

checkpoint1 = ModelCheckpoint(
    filepath='ft2_model.keras', 
    monitor='val_loss',       
    save_best_only=True, 
    verbose=1)

history1 = frozen_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, checkpoint1])

In [ ]:
# Evaluate
final_model = tf.keras.models.load_model("ft2_model.keras") #loading the final fine-tuned model

y_true= []
y_pred= []
for images, labels in val_ds:
    predictions= final_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

In [ ]:
# Confusion Martix
cm= confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels= train_ds.class_names,
            yticklabels= train_ds.class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Plant Disease Confusion Matrix")
plt.show()

In [ ]:
# Classification Report
print(
    classification_report(
        y_true,
        y_pred,
        target_names=train_ds.class_names
    )
)